In [1]:
import sys, os

# Ensure project_root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))  # if notebook is in notebooks/
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")  # Must end in edtech-nlp-pipeline

Working directory: c:\Users\yashg\edtech-nlp-pipeline


In [2]:
import warnings
warnings.filterwarnings('ignore')   # Suppress DeBERTa sentencepiece byte-fallback warning

import os, torch, transformers

print(f"Working directory  : {os.getcwd()}")
print(f"transformers       : {transformers.__version__}")  # Must be 4.44.2
print(f"torch              : {torch.__version__}")
print(f"CUDA available     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                : {torch.cuda.get_device_name(0)}")
    print(f"VRAM               : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU. Training will be extremely slow on CPU.")

# Uncomment if transformers version is wrong:
# !pip install transformers==4.44.2 huggingface_hub==0.24.6

from src.utils.config import load_config
cfg = load_config("configs/config.yaml")
print(f"Config loaded. stage1 keys: {list(cfg['stage1'].keys())}")
assert cfg['stage1']['num_labels'] == 13, "num_labels must be 13!"

Working directory  : c:\Users\yashg\edtech-nlp-pipeline
transformers       : 4.38.2
torch              : 2.2.0+cu121
CUDA available     : True
GPU                : NVIDIA GeForce RTX 4050 Laptop GPU
VRAM               : 6.4 GB
Config loaded. stage1 keys: ['train_path', 'external_records_path', 'val_fraction', 'random_state', 'model_name', 'max_length', 'dropout', 'num_labels', 'num_epochs', 'batch_size', 'learning_rate', 'warmup_fraction', 'max_grad_norm', 'class_weight_cap', 'max_external_records', 'inference_threshold', 'model_save_dir']


In [3]:
from src.data.pii_dataset import (
    load_pii_records, train_val_split, PIIDataset, LABEL2ID, ID2LABEL
)

records    = load_pii_records(cfg['stage1']['train_path'])
train_recs, val_recs = train_val_split(
    records,
    val_fraction=cfg['stage1']['val_fraction'],
    random_state=cfg['stage1']['random_state']
)
print(f"Competition train: {len(train_recs)} docs | Val: {len(val_recs)} docs")
print(f"num_labels: {len(LABEL2ID)}")    # Must be 13

Competition train: 5446 docs | Val: 1361 docs
num_labels: 13


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    cfg['stage1']['model_name'], use_fast=True
)
# Val dataset: competition data only — do NOT add external records here
val_ds = PIIDataset(val_recs, tokenizer, LABEL2ID,
                    max_length=cfg['stage1']['max_length'])
print(f"Val samples: {len(val_ds)}")

Val samples: 1361


In [5]:
from src.models.pii_model import PIITokenClassifier
from src.training.pii_trainer import compute_class_weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = PIITokenClassifier(
    cfg['stage1']['model_name'],
    num_labels=len(LABEL2ID),
    dropout=cfg['stage1']['dropout']
).to(device)
model.enable_gradient_checkpointing()

# Smoke-test weights — competition data only
smoke_weights = compute_class_weights(train_recs, LABEL2ID, device,
                                      max_weight=cfg['stage1']['class_weight_cap'])
print("Class weights (smoke test):")
from src.data.pii_dataset import PII_LABELS
for lbl, w in zip(PII_LABELS, smoke_weights.cpu().tolist()):
    print(f"  {lbl:<25} {w:.3f}")

Class weights (smoke test):
  O                         0.100
  B-NAME_STUDENT            6.673
  I-NAME_STUDENT            6.870
  B-EMAIL                   10.634
  B-USERNAME                12.532
  B-ID_NUM                  9.519
  I-ID_NUM                  13.630
  B-PHONE_NUM               12.532
  I-PHONE_NUM               11.838
  B-URL_PERSONAL            9.199
  I-URL_PERSONAL            13.630
  B-STREET_ADDRESS          13.630
  I-STREET_ADDRESS          11.328


In [6]:
from datasets import load_dataset

print("Loading ai4privacy/pii-masking-400k in streaming mode...")
pii_extra = load_dataset(
    "ai4privacy/pii-masking-400k",
    split="train",
    streaming=True        # ← reads records one at a time, no RAM spike
)

# Inspect structure from first record only
first = next(iter(pii_extra))
print(f"Keys: {list(first.keys())}")
print(f"Sample tokens: {first['mbert_tokens'][:10]}")
print(f"Sample labels: {first['mbert_token_classes'][:10]}")
import os, sys
PROJECT_ROOT = r"c:\Users\yashg\edtech-nlp-pipeline"
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

Loading ai4privacy/pii-masking-400k in streaming mode...
Keys: ['source_text', 'locale', 'language', 'split', 'privacy_mask', 'uid', 'masked_text', 'mbert_tokens', 'mbert_token_classes']
Sample tokens: ['<', 'p', '>', 'My', 'child', 'fa', '##oz', '##zs', '##d', '##3']
Sample labels: ['O', 'O', 'O', 'O', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME']


In [7]:
import hashlib, json

def text_hash(text: str) -> str:
    return hashlib.md5(text.strip().lower().encode()).hexdigest()

with open("data/raw/pii/train.json") as f:
    competition_data = json.load(f)

competition_hashes = {text_hash(doc['full_text']) for doc in competition_data}
print(f"Competition hashes computed: {len(competition_hashes)}")

# Check first 10000 external records
overlap  = 0
checked  = 0
for rec in pii_extra:
    if checked >= 10_000:
        break
    ext_text = " ".join(rec['mbert_tokens'])   # ← streaming field name
    if text_hash(ext_text) in competition_hashes:
        overlap += 1
    checked += 1

print(f"Checked: {checked} records")
print(f"Overlap: {overlap} documents")
if overlap > 0:
    print("⚠️  Overlap found. Do NOT use external data.")
    raise ValueError("External data overlaps with competition data.")
else:
    print("✓ External data is safe to use.")

Competition hashes computed: 6807
Checked: 10000 records
Overlap: 0 documents
✓ External data is safe to use.


In [8]:
import json, random
from collections import Counter
from datasets import load_dataset
from src.utils.config import load_config
from src.data.pii_dataset import LABEL2ID

cfg = load_config("configs/config.yaml")

EXTERNAL_LABEL_MAP = {
    'FIRSTNAME': 'NAME_STUDENT', 'LASTNAME': 'NAME_STUDENT',
    'MIDDLENAME': 'NAME_STUDENT', 'EMAIL': 'EMAIL',
    'USERNAME': 'USERNAME',
    'TELEPHONENUM': 'PHONE_NUM',
    'ZIPCODE': 'STREET_ADDRESS',
    'STREETADDRESS': 'STREET_ADDRESS', 'URL': 'URL_PERSONAL',
    'IDCARD': 'ID_NUM', 'DRIVERLICENSE': 'ID_NUM',
    'PASSPORT': 'ID_NUM', 'SOCIALNUMBER': 'ID_NUM',
    'TAXNUM': 'ID_NUM', 'ACCOUNTNUM': 'ID_NUM', 'IBAN': 'ID_NUM',
}
TARGET_RARE = {
    'B-STREET_ADDRESS', 'B-PHONE_NUM', 'B-USERNAME',
    'I-URL_PERSONAL', 'I-ID_NUM'
}

def map_label(raw: str) -> str:
    if raw == 'O' or '-' not in raw:
        return 'O'
    prefix, etype = raw.split('-', 1)
    mapped = EXTERNAL_LABEL_MAP.get(etype)
    if not mapped:
        return 'O'
    candidate = f"{prefix}-{mapped}"
    return candidate if candidate in LABEL2ID else 'O'

def merge_subword_tokens(subword_tokens: list, subword_labels: list) -> tuple:
    words, word_labels = [], []
    current_word, current_label = None, None
    for token, label in zip(subword_tokens, subword_labels):
        if token.startswith('##'):
            if current_word is not None:
                current_word += token[2:]
        else:
            if current_word is not None:
                words.append(current_word)
                word_labels.append(current_label)
            current_word  = token
            current_label = label
    if current_word is not None:
        words.append(current_word)
        word_labels.append(current_label)
    return words, word_labels

# ── Step 1: Load dataset ──────────────────────────────────────────────────────
print("Loading ai4privacy dataset (streaming)...")
pii_extra = load_dataset(
    "ai4privacy/pii-masking-400k", split="train", streaming=True
)
print("Stream ready.")

MAX_RECORDS = cfg['stage1']['max_external_records']

# ── Step 2: Build rare + common pools ────────────────────────────────────────
rare_records, common_records = [], []
for i, rec in enumerate(pii_extra):
    lang = rec.get('language', 'en')
    if lang not in ('en', 'English', None, ''):
        continue

    words, raw_word_labels = merge_subword_tokens(
        rec['mbert_tokens'], rec['mbert_token_classes']
    )
    mapped_labels = [map_label(lbl) for lbl in raw_word_labels]

    if all(lbl == 'O' for lbl in mapped_labels):
        continue

    converted = {
        'document': f'ext_{i}',
        'tokens':   words,
        'labels':   mapped_labels
    }
    if any(lbl in TARGET_RARE for lbl in mapped_labels):
        rare_records.append(converted)
    else:
        common_records.append(converted)

    if len(rare_records) >= MAX_RECORDS // 2 and \
       len(common_records) >= MAX_RECORDS // 2:
        break

random.seed(42)
random.shuffle(rare_records)
random.shuffle(common_records)
n_rare   = min(len(rare_records),   MAX_RECORDS // 2)
n_common = min(len(common_records), MAX_RECORDS - n_rare)
external_records = rare_records[:n_rare] + common_records[:n_common]
print(f"\nBefore caps : {len(external_records)} records")

# ── Step 3: Compute competition label counts (needed for all caps) ────────────
from src.data.pii_dataset import load_pii_records, train_val_split
records    = load_pii_records(cfg['stage1']['train_path'])
train_recs, _ = train_val_split(records,
                                 val_fraction=cfg['stage1']['val_fraction'],
                                 random_state=cfg['stage1']['random_state'])
comp_counter = Counter(lbl for r in train_recs for lbl in r['labels'])

# ── Step 4: URL cap (3× competition) ─────────────────────────────────────────
URL_CAP  = comp_counter.get('B-URL_PERSONAL', 0) * 3
url_seen, capped = 0, []
for rec in external_records:
    b_url = sum(1 for lbl in rec['labels'] if lbl == 'B-URL_PERSONAL')
    if b_url > 0:
        if url_seen + b_url <= URL_CAP:
            capped.append(rec)
            url_seen += b_url
    else:
        capped.append(rec)
print(f"URL cap      : {url_seen} / {URL_CAP} | records: {len(capped)}")
external_records = capped

# ── Step 5: USERNAME cap (3× competition) ────────────────────────────────────
USERNAME_CAP = comp_counter.get('B-USERNAME', 0) * 3
user_seen, capped = 0, []
for rec in external_records:
    b_user = sum(1 for lbl in rec['labels'] if lbl == 'B-USERNAME')
    if b_user > 0:
        if user_seen + b_user <= USERNAME_CAP:
            capped.append(rec)
            user_seen += b_user
    else:
        capped.append(rec)
print(f"USERNAME cap : {user_seen} / {USERNAME_CAP} | records: {len(capped)}")
external_records = capped

# ── Step 6: NAME_STUDENT cap (5× competition) ────────────────────────────────
NAME_CAP = comp_counter.get('B-NAME_STUDENT', 0) * 5
name_seen, capped = 0, []
for rec in external_records:
    b_name = sum(1 for lbl in rec['labels'] if lbl == 'B-NAME_STUDENT')
    if b_name > 0:
        if name_seen + b_name <= NAME_CAP:
            capped.append(rec)
            name_seen += b_name
    else:
        capped.append(rec)
print(f"NAME cap     : {name_seen} / {NAME_CAP} | records: {len(capped)}")
external_records = capped

# ── Step 7: Save ─────────────────────────────────────────────────────────────
save_path = cfg['stage1']['external_records_path']
with open(save_path, 'w', encoding='utf-8') as f:
    json.dump(external_records, f, ensure_ascii=False)
print(f"\nFinal records saved : {len(external_records)} → {save_path}")

Loading ai4privacy dataset (streaming)...
Stream ready.

Before caps : 5000 records
URL cap      : 0 / 252 | records: 5000
USERNAME cap : 9 / 9 | records: 3998
NAME cap     : 0 / 5255 | records: 3998

Final records saved : 3998 → data/raw/pii/external_records.json


In [9]:
import json
from collections import Counter
from src.data.pii_dataset import LABEL2ID

with open(cfg['stage1']['external_records_path'], encoding='utf-8') as f:
    ext_check = json.load(f)

print(f"External records loaded: {len(ext_check)}")
print(f"\nSample record:")
print(f"  tokens : {ext_check[0]['tokens'][:8]}")
print(f"  labels : {ext_check[0]['labels'][:8]}")

# Label distribution in external records
ext_counter = Counter()
for rec in ext_check:
    ext_counter.update(rec['labels'])

print(f"\nLabel distribution in external records:")
for lbl, cnt in sorted(ext_counter.items(), key=lambda x: -x[1]):
    if lbl != 'O':
        print(f"  {lbl:<25} {cnt}")

# Confirm rare labels are now represented
for rare_label in ['B-STREET_ADDRESS', 'B-PHONE_NUM', 'B-USERNAME']:
    count = ext_counter.get(rare_label, 0)
    status = "✓" if count > 50 else "⚠️ low"
    print(f"\n{status}  {rare_label}: {count} examples in external data")

External records loaded: 3998

Sample record:
  tokens : ['Application', 'for', 'Didcot', 'educational', 'aid', 'by', 'Mozilla', '/']
  labels : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Label distribution in external records:
  I-PHONE_NUM               3393
  B-EMAIL                   1918
  B-ID_NUM                  1361
  I-ID_NUM                  795
  B-PHONE_NUM               758
  B-STREET_ADDRESS          610
  I-STREET_ADDRESS          400
  B-USERNAME                9

✓  B-STREET_ADDRESS: 610 examples in external data

✓  B-PHONE_NUM: 758 examples in external data

⚠️ low  B-USERNAME: 9 examples in external data


In [10]:
# Load external records saved by T9
import json
import torch
from src.models.pii_model import PIITokenClassifier
from src.training.pii_trainer import compute_class_weights, run_training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(cfg['stage1']['external_records_path'], encoding='utf-8') as f:
    external_records = json.load(f)

print(f"Competition train : {len(train_recs)} docs")
print(f"External records  : {len(external_records)} docs")

# Merge: competition + external for training only
# Val stays competition-only
augmented_train_recs = train_recs + external_records
print(f"Augmented train   : {len(augmented_train_recs)} docs")

augmented_train_ds = PIIDataset(augmented_train_recs, tokenizer, LABEL2ID,
                                max_length=cfg['stage1']['max_length'])

# Class weights from merged data
augmented_weights = compute_class_weights(
    augmented_train_recs, LABEL2ID, device,
    max_weight=cfg['stage1']['class_weight_cap']
)

# Reinitialise model from scratch (not from smoke test weights)
model = PIITokenClassifier(
    cfg['stage1']['model_name'],
    num_labels=len(LABEL2ID),
    dropout=cfg['stage1']['dropout']
).to(device)
model.enable_gradient_checkpointing()

best_f1 = run_training(model, augmented_train_ds, val_ds,
                        cfg['stage1'], device, augmented_weights)
print(f"\nFull run best entity F1: {best_f1:.4f}")

Competition train : 5446 docs
External records  : 3998 docs
Augmented train   : 9444 docs
Epoch 1/5  loss=0.3125  entity_F1=0.7864  P=0.6846  R=0.9236
  → Saved best model (epoch 1)
Epoch 2/5  loss=0.0228  entity_F1=0.8054  P=0.6881  R=0.9709
  → Saved best model (epoch 2)
Epoch 3/5  loss=0.0102  entity_F1=0.8386  P=0.7730  R=0.9164
  → Saved best model (epoch 3)
Epoch 4/5  loss=0.0039  entity_F1=0.8971  P=0.8365  R=0.9673
  → Saved best model (epoch 4)
Epoch 5/5  loss=0.0018  entity_F1=0.8986  P=0.8391  R=0.9673
  → Saved best model (epoch 5)

Best entity F1: 0.8986 at epoch 5

Full run best entity F1: 0.8986


In [11]:
import pandas as pd
from datetime import date

log_path = "outputs/experiment_log.csv"
log = pd.read_csv(log_path, encoding='latin-1')
new_row = {
    'experiment_id': 'exp_004',
    'date':          str(date.today()),
    'model':         cfg['stage1']['model_name'],
    'stage':         'stage1_pii',
    'hypothesis':    'Baseline DeBERTa-v3-base, 13 labels, log-scaled weights, +external data',
    'change_made':   'First full run with augmented training set',
    'metric_before': 'N/A',
    'metric_after':  f'entity_f1={best_f1:.4f}',
    'verdict':       'baseline',
    'notes':         (f"train={len(augmented_train_recs)} docs "
                      f"(competition={len(train_recs)} + external={len(external_records)}), "
                      f"val={len(val_recs)} docs (competition only), "
                      f"max_length={cfg['stage1']['max_length']}, "
                      f"epochs={cfg['stage1']['num_epochs']}, "
                      f"batch_size={cfg['stage1']['batch_size']}")
}
log = pd.concat([log, pd.DataFrame([new_row])], ignore_index=True)
log.to_csv(log_path, index=False)
print("Experiment logged.")
print(log.tail(3)[['experiment_id', 'stage', 'metric_after', 'verdict']])

Experiment logged.
  experiment_id       stage      metric_after   verdict
4       exp_004  stage1_pii  entity_f1=0.8230  baseline
5       exp_004  stage1_pii  entity_f1=0.8271  baseline
6       exp_004  stage1_pii  entity_f1=0.8986  baseline


In [12]:
from src.evaluation.pii_metrics import compute_entity_f1
from src.training.pii_trainer import evaluate as run_eval
from torch.utils.data import DataLoader

val_loader = DataLoader(
    val_ds, batch_size=cfg['stage1']['batch_size'],
    shuffle=False, num_workers=0
)
all_true, all_pred = run_eval(model, val_loader, device, ID2LABEL)
metrics = compute_entity_f1(all_true, all_pred)

print(metrics['report'])
print(f"\nEntity F1        : {metrics['entity_f1']:.4f}")
print(f"Entity Precision : {metrics['entity_precision']:.4f}")
print(f"Entity Recall    : {metrics['entity_recall']:.4f}")

                precision    recall  f1-score   support

         EMAIL       1.00      1.00      1.00        16
        ID_NUM       0.64      0.82      0.72        17
  NAME_STUDENT       0.87      0.98      0.92       227
     PHONE_NUM       1.00      1.00      1.00         3
STREET_ADDRESS       0.00      0.00      0.00         1
  URL_PERSONAL       0.56      0.90      0.69        10
      USERNAME       0.33      1.00      0.50         1

     micro avg       0.84      0.97      0.90       275
     macro avg       0.63      0.82      0.69       275
  weighted avg       0.85      0.97      0.90       275


Entity F1        : 0.8986
Entity Precision : 0.8391
Entity Recall    : 0.9673


In [13]:
from src.evaluation.pii_metrics import apply_entity_threshold, compute_entity_f1
from src.training.pii_trainer import evaluate as run_eval
from torch.utils.data import DataLoader

val_loader = DataLoader(val_ds, batch_size=cfg['stage1']['batch_size'],
                        shuffle=False, num_workers=0)
all_true, _, all_logits = run_eval(model, val_loader, device,
                                    ID2LABEL, return_logits=True)

print(f"{'NAME_fl':>8} {'URL_fl':>8} {'USER_fl':>8} {'P':>8} {'R':>8} {'F1':>8}")
print("-" * 58)

best_f1, best_floors = 0.0, {}

for name_floor in [0.45, 0.50, 0.55]:
    for url_floor in [0.60, 0.70, 0.80]:
        for user_floor in [0.70, 0.80, 0.90]:
            floors = {
                "B-NAME_STUDENT": name_floor,
                "I-NAME_STUDENT": name_floor,
                "B-URL_PERSONAL": url_floor,
                "I-URL_PERSONAL": url_floor,
                "B-USERNAME":     user_floor,
                "I-USERNAME":     user_floor,
                "B-ID_NUM":       0.55,
                "I-ID_NUM":       0.55,
            }
            preds = [apply_entity_threshold(l, ID2LABEL, entity_floors=floors)
                     for l in all_logits]
            r = compute_entity_f1(all_true, preds)

            if r['entity_f1'] > best_f1:
                best_f1     = r['entity_f1']
                best_floors = floors.copy()

            print(f"{name_floor:>8.2f} {url_floor:>8.2f} {user_floor:>8.2f}  "
                  f"{r['entity_precision']:>8.3f}  "
                  f"{r['entity_recall']:>8.3f}  "
                  f"{r['entity_f1']:>8.3f}")

print(f"\nBest F1     : {best_f1:.4f}")
print(f"Best floors : {best_floors}")

 NAME_fl   URL_fl  USER_fl        P        R       F1
----------------------------------------------------------
    0.45     0.60     0.70     0.839     0.967     0.899
    0.45     0.60     0.80     0.839     0.967     0.899
    0.45     0.60     0.90     0.842     0.967     0.900
    0.45     0.70     0.70     0.842     0.967     0.900
    0.45     0.70     0.80     0.842     0.967     0.900
    0.45     0.70     0.90     0.844     0.967     0.902
    0.45     0.80     0.70     0.844     0.967     0.902
    0.45     0.80     0.80     0.844     0.967     0.902
    0.45     0.80     0.90     0.847     0.967     0.903
    0.50     0.60     0.70     0.839     0.967     0.899
    0.50     0.60     0.80     0.839     0.967     0.899
    0.50     0.60     0.90     0.842     0.967     0.900
    0.50     0.70     0.70     0.842     0.967     0.900
    0.50     0.70     0.80     0.842     0.967     0.900
    0.50     0.70     0.90     0.844     0.967     0.902
    0.50     0.80     0.70     0

In [14]:
import yaml

with open("configs/config.yaml", "r") as f:
    full_cfg = yaml.safe_load(f)

full_cfg['stage1']['entity_floors'] = {
    'B-NAME_STUDENT': 0.55,
    'I-NAME_STUDENT': 0.55,
    'B-URL_PERSONAL': 0.80,
    'I-URL_PERSONAL': 0.80,
    'B-USERNAME':     0.90,
    'I-USERNAME':     0.90,
    'B-ID_NUM':       0.55,
    'I-ID_NUM':       0.55,
}
full_cfg['stage1']['inference_threshold'] = 0.5

with open("configs/config.yaml", "w") as f:
    yaml.dump(full_cfg, f, default_flow_style=False)

print("Floors saved to config.yaml")

Floors saved to config.yaml


In [15]:
# Cell T7-2 — Collect logits
all_true, _, all_logits = run_eval(
    model, val_loader, device, ID2LABEL, return_logits=True
)
print(f"Sequences collected : {len(all_logits)}")

Sequences collected : 1361


In [16]:
# Cell T7-4 — Lock best floors into config.yaml
import yaml

with open("configs/config.yaml", "r") as f:
    full_cfg = yaml.safe_load(f)

full_cfg['stage1']['entity_floors']       = best_floors
full_cfg['stage1']['inference_threshold'] = 0.5

with open("configs/config.yaml", "w") as f:
    yaml.dump(full_cfg, f, default_flow_style=False)

# Verify
cfg = load_config("configs/config.yaml")
print("Saved floors:")
for k, v in cfg['stage1']['entity_floors'].items():
    print(f"  {k:<25} {v}")
print(f"\ninference_threshold : {cfg['stage1']['inference_threshold']}")
print("Config saved OK")

Saved floors:
  B-ID_NUM                  0.55
  B-NAME_STUDENT            0.55
  B-URL_PERSONAL            0.8
  B-USERNAME                0.9
  I-ID_NUM                  0.55
  I-NAME_STUDENT            0.55
  I-URL_PERSONAL            0.8
  I-USERNAME                0.9

inference_threshold : 0.5
Config saved OK


In [17]:
# Cell T7-5 — Error analysis: false negatives by entity type
from collections import defaultdict

# Use best floors for final predictions
best_preds = [
    apply_entity_threshold(l, ID2LABEL,
                            entity_floors=cfg['stage1']['entity_floors'])
    for l in all_logits
]

# False negatives — PII the model missed
missed = defaultdict(int)
for true_seq, pred_seq in zip(all_true, best_preds):
    for t_label, p_label in zip(true_seq, pred_seq):
        if t_label != 'O' and p_label == 'O':
            missed[t_label] += 1

# False positives — things wrongly flagged as PII
wrong = defaultdict(int)
for true_seq, pred_seq in zip(all_true, best_preds):
    for t_label, p_label in zip(true_seq, pred_seq):
        if t_label == 'O' and p_label != 'O':
            wrong[p_label] += 1

print("False negatives (missed PII) by entity type:")
for label, count in sorted(missed.items(), key=lambda x: -x[1]):
    print(f"  {label:<25} {count}")

print("\nFalse positives (over-fired) by entity type:")
for label, count in sorted(wrong.items(), key=lambda x: -x[1]):
    print(f"  {label:<25} {count}")

False negatives (missed PII) by entity type:
  I-STREET_ADDRESS          8
  B-ID_NUM                  2
  B-NAME_STUDENT            1
  I-NAME_STUDENT            1
  B-URL_PERSONAL            1
  B-STREET_ADDRESS          1

False positives (over-fired) by entity type:
  B-NAME_STUDENT            30
  I-NAME_STUDENT            14
  B-ID_NUM                  6
  B-URL_PERSONAL            5
  B-USERNAME                1


In [18]:
# Cell T7-6 — Final val report with best floors applied
final_metrics = compute_entity_f1(all_true, best_preds)
print(final_metrics['report'])
print(f"Entity F1        : {final_metrics['entity_f1']:.4f}")
print(f"Entity Precision : {final_metrics['entity_precision']:.4f}")
print(f"Entity Recall    : {final_metrics['entity_recall']:.4f}")

                precision    recall  f1-score   support

         EMAIL       1.00      1.00      1.00        16
        ID_NUM       0.64      0.82      0.72        17
  NAME_STUDENT       0.87      0.98      0.93       227
     PHONE_NUM       1.00      1.00      1.00         3
STREET_ADDRESS       0.00      0.00      0.00         1
  URL_PERSONAL       0.64      0.90      0.75        10
      USERNAME       0.50      1.00      0.67         1

     micro avg       0.85      0.97      0.90       275
     macro avg       0.66      0.82      0.72       275
  weighted avg       0.86      0.97      0.91       275

Entity F1        : 0.9048
Entity Precision : 0.8498
Entity Recall    : 0.9673


In [20]:
# Cell T7-7 — Log to experiment_log.csv
import pandas as pd
from datetime import date

log_path = "outputs/experiment_log.csv"
log = pd.read_csv(log_path, encoding='utf-8-sig')

new_row = {
    'experiment_id': 'exp_006',
    'date':          str(date.today()),
    'model':         cfg['stage1']['model_name'],
    'stage':         'stage1_pii',
    'hypothesis':    'Entity floor sweep to fix URL/USERNAME over-firing',
    'change_made':   f"entity_floors applied: {cfg['stage1']['entity_floors']}",
    'metric_before': f"F1=0.8986  P=0.8391  R=0.9673",
    'metric_after':  f"F1={final_metrics['entity_f1']:.4f}  "
                     f"P={final_metrics['entity_precision']:.4f}  "
                     f"R={final_metrics['entity_recall']:.4f}",
    'verdict':       'improvement',
    'notes':         (f"Best floors from sweep. "
                      f"val={len(val_recs)} docs (competition only). "
                      f"No retrain — inference-time fix only.")
}
log = pd.concat([log, pd.DataFrame([new_row])], ignore_index=True)
log.to_csv(log_path, index=False, encoding='utf-8-sig')
print("Logged exp_006.")
print(log[log['stage'] == 'stage1_pii'][
    ['experiment_id', 'metric_after', 'verdict']
].to_string())

Logged exp_006.
  experiment_id                   metric_after      verdict
0       PII-000                            TBD          TBD
3       exp_004               entity_f1=0.8448     baseline
4       exp_004               entity_f1=0.8230     baseline
5       exp_004               entity_f1=0.8271     baseline
6       exp_004               entity_f1=0.8986     baseline
7       exp_006  F1=0.9048  P=0.8498  R=0.9673  improvement


In [22]:
# Save tokenizer to model directory — one-time fix
from transformers import AutoTokenizer
from src.utils.config import load_config

cfg       = load_config("configs/config.yaml")
tokenizer = AutoTokenizer.from_pretrained(
    cfg['stage1']['model_name'], use_fast=True
)
tokenizer.save_pretrained(cfg['stage1']['model_save_dir'])
print("Tokenizer saved.")

Tokenizer saved.


In [23]:
tokenizer.save_pretrained(cfg['stage1']['model_save_dir'])
print("Tokenizer saved.")

Tokenizer saved.


In [24]:
# Cell T8-1 — Verify saved files exist
import os

save_dir  = cfg['stage1']['model_save_dir']
required  = [
    'classifier_head.pt',
    'config.json',
    'tokenizer.json',
    'tokenizer_config.json',
    'spm.model',
    'special_tokens_map.json',
]
print(f"Checking {save_dir}/\n")
all_present = True
for fname in required:
    path   = os.path.join(save_dir, fname)
    status = "✓" if os.path.exists(path) else "✗ MISSING"
    print(f"  {status}  {fname}")
    if not os.path.exists(path):
        all_present = False

if all_present:
    print("\nAll files present.")
else:
    print("\n⚠️  Missing files — best model checkpoint never saved.")
    print("Check that entity F1 improved during training (run_training saves on improvement only).")

Checking outputs/models/pii/

  ✓  classifier_head.pt
  ✓  config.json
  ✓  tokenizer.json
  ✓  tokenizer_config.json
  ✓  spm.model
  ✓  special_tokens_map.json

All files present.


In [25]:
# Cell T8-2 — Load from disk and verify inference
# Tests the exact loading path Phase 3 (Streamlit) will use.
# No reference to any in-memory objects from training.
import os, sys, torch, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = r"c:\Users\yashg\edtech-nlp-pipeline"
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.config import load_config
from src.data.pii_dataset import (
    load_pii_records, train_val_split, PIIDataset, LABEL2ID, ID2LABEL
)
from src.models.pii_model import PIITokenClassifier
from src.evaluation.pii_metrics import apply_entity_threshold
from transformers import AutoTokenizer

cfg_fresh = load_config("configs/config.yaml")
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load backbone + head from disk
print(f"Loading from : {cfg_fresh['stage1']['model_save_dir']}")
model_loaded = PIITokenClassifier.from_saved(
    cfg_fresh['stage1']['model_save_dir'],
    num_labels=len(LABEL2ID),
    dropout=cfg_fresh['stage1']['dropout']
).to(device).eval()

# Rebuild val dataset from scratch
tokenizer_check = AutoTokenizer.from_pretrained(
    cfg_fresh['stage1']['model_name'], use_fast=True
)
records_check   = load_pii_records(cfg_fresh['stage1']['train_path'])
_, val_recs_check = train_val_split(
    records_check,
    val_fraction=cfg_fresh['stage1']['val_fraction'],
    random_state=cfg_fresh['stage1']['random_state']
)
val_ds_check = PIIDataset(
    val_recs_check[:1], tokenizer_check, LABEL2ID,
    max_length=cfg_fresh['stage1']['max_length']
)

# Forward pass
sample = val_ds_check[0]
with torch.no_grad():
    out = model_loaded(
        input_ids      = sample['input_ids'].unsqueeze(0).to(device),
        attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
    )

print(f"Logits shape : {out['logits'].shape}")
assert out['logits'].shape == (
    1,
    cfg_fresh['stage1']['max_length'],
    len(LABEL2ID)
), "Shape mismatch — check num_labels"
print("Forward pass : OK")

# Verify entity floors are in config
assert 'entity_floors' in cfg_fresh['stage1'], \
    "entity_floors missing from config — re-run T7 Cell T7-4"
print(f"Entity floors: {cfg_fresh['stage1']['entity_floors']}")

# Verify inference with floors applied
import numpy as np
logits_np = out['logits'].squeeze(0).cpu().float().numpy()
preds     = apply_entity_threshold(
    logits_np,
    ID2LABEL,
    entity_floors=cfg_fresh['stage1']['entity_floors']
)
print(f"Sample predictions (first 10 tokens): {preds[:10]}")
print("Load + inference with floors : OK")

Loading from : outputs/models/pii
Logits shape : torch.Size([1, 512, 13])
Forward pass : OK
Entity floors: {'B-ID_NUM': 0.55, 'B-NAME_STUDENT': 0.55, 'B-URL_PERSONAL': 0.8, 'B-USERNAME': 0.9, 'I-ID_NUM': 0.55, 'I-NAME_STUDENT': 0.55, 'I-URL_PERSONAL': 0.8, 'I-USERNAME': 0.9}
Sample predictions (first 10 tokens): ['B-PHONE_NUM', 'B-NAME_STUDENT', 'I-NAME_STUDENT', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Load + inference with floors : OK


In [33]:
import pandas as pd

log = pd.read_csv("outputs/experiment_log.csv", encoding='utf-8-sig')

print(f"Total experiment rows: {len(log)}")
print()
print(log[['experiment_id', 'metric_after', 'verdict']].to_string())

# Check Phase 1 rows exist by experiment_id
phase1_ids = ['exp_004', 'exp_005', 'exp_006']
found_ids  = log['experiment_id'].tolist()
missing    = [eid for eid in phase1_ids if eid not in found_ids]

if missing:
    print(f"\n⚠️  Missing experiment IDs: {missing}")
    print("Re-run the logging cells for those experiments.")
else:
    print("\n✓  All Phase 1 experiment rows present.")
    print("Experiment log OK")

Total experiment rows: 5

  experiment_id                   metric_after      verdict
0       PII-000                            TBD          TBD
1     ESSAY-000                            TBD          TBD
2       exp_006  F1=0.9048  P=0.8498  R=0.9673  improvement
3       exp_005          external_records=3998    data_prep
4       exp_004               entity_f1=0.8986     baseline

✓  All Phase 1 experiment rows present.
Experiment log OK


In [29]:
import pandas as pd
from datetime import date
from src.utils.config import load_config
import json

cfg      = load_config("configs/config.yaml")
log_path = "outputs/experiment_log.csv"
log      = pd.read_csv(log_path, encoding='utf-8-sig')

with open(cfg['stage1']['external_records_path'], encoding='utf-8') as f:
    ext_check = json.load(f)

new_row = {
    'experiment_id': 'exp_005',
    'date':          str(date.today()),
    'model':         cfg['stage1']['model_name'],
    'stage':         'stage1_pii',
    'hypothesis':    'Add AI4Privacy external data to boost rare label coverage',
    'change_made':   (f'+{len(ext_check)} AI4Privacy records. '
                      f'Caps: URL=1x, USERNAME=3x, NAME removed entirely.'),
    'metric_before': 'N/A (pre-training)',
    'metric_after':  f'external_records={len(ext_check)}',
    'verdict':       'data_prep',
    'notes':         ('rare_labels_targeted=STREET_ADDRESS,PHONE_NUM,USERNAME,'
                      'URL_PERSONAL,ID_NUM. Leakage check: overlap=0.')
}
log = pd.concat([log, pd.DataFrame([new_row])], ignore_index=True)
log.to_csv(log_path, index=False, encoding='utf-8-sig')
print("exp_005 logged.")

exp_005 logged.


In [30]:
log      = pd.read_csv(log_path, encoding='utf-8-sig')

# Keep last exp_004 row (most recent = final retrain result) + all others
is_004   = log['experiment_id'] == 'exp_004'
last_004 = log[is_004].tail(1)
rest     = log[~is_004]
log      = pd.concat([rest, last_004], ignore_index=True).sort_index()

log.to_csv(log_path, index=False, encoding='utf-8-sig')
print("Duplicates cleaned.")
print(log[['experiment_id', 'metric_after', 'verdict']].to_string())

Duplicates cleaned.
  experiment_id                   metric_after      verdict
0       PII-000                            TBD          TBD
1     ESSAY-000                            TBD          TBD
2       exp_006  F1=0.9048  P=0.8498  R=0.9673  improvement
3       exp_005          external_records=3998    data_prep
4       exp_004               entity_f1=0.8986     baseline


In [34]:
# Cell T8-4 — Phase 1 done checklist
import os
from src.utils.config import load_config

cfg   = load_config("configs/config.yaml")
s1    = cfg['stage1']
checks = [
    (
        os.path.exists(os.path.join(s1['model_save_dir'], 'classifier_head.pt')),
        "classifier_head.pt exists"
    ),
    (
        os.path.exists(os.path.join(s1['model_save_dir'], 'config.json')),
        "backbone config.json exists"
    ),
    (
        os.path.exists(s1['external_records_path']),
        "external_records.json exists"
    ),
    (
        s1['num_labels'] == 13,
        f"num_labels == 13 (got {s1['num_labels']})"
    ),
    (
        'entity_floors' in s1,
        "entity_floors present in config"
    ),
    (
        s1.get('inference_threshold') == 0.5,
        f"inference_threshold == 0.5 (got {s1.get('inference_threshold')})"
    ),
    (
        len(pd.read_csv('outputs/experiment_log.csv',
                        encoding='latin-1')
              [lambda df: df['stage'] == 'stage1_pii']) >= 3,
        "experiment_log has ≥3 Phase 1 rows"
    ),
]

print("Phase 1 Done Checklist")
print("=" * 45)
all_passed = True
for passed, label in checks:
    status = "✓" if passed else "✗ FAILED"
    print(f"  {status}  {label}")
    if not passed:
        all_passed = False

print("=" * 45)
if all_passed:
    print("ALL CHECKS PASSED — Phase 1 complete. Ready for Phase 3.")
else:
    print("⚠️  Fix failing checks before moving to Phase 3.")

Phase 1 Done Checklist
  ✓  classifier_head.pt exists
  ✓  backbone config.json exists
  ✓  external_records.json exists
  ✓  num_labels == 13 (got 13)
  ✓  entity_floors present in config
  ✓  inference_threshold == 0.5 (got 0.5)
  ✓  experiment_log has ≥3 Phase 1 rows
ALL CHECKS PASSED — Phase 1 complete. Ready for Phase 3.
